In [5]:
import sys
import os
import logging
import gc
import time
import torch
import warnings
import psutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as pltSAvFRAb6tEi9UcN
from gliner import GLiNER
from gliner.data_processing.collator import DataCollator
from gliner.training import Trainer, TrainingArguments
from transformers import TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType,PeftModel

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
sys.path

/opt/app/notebooks/abhishek/active_gliner/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python311.zip',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11/lib-dynload',
 '',
 '/opt/app/notebooks/abhishek/active_gliner/.venv/lib/python3.11/site-packages',
 '/tmp/tmpxous61wc']

In [6]:
src_path=os.path.join(os.path.dirname(os.getcwd()),'src')
sys.path.append(src_path)
sys.path


['/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python311.zip',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11',
 '/root/.local/share/uv/python/cpython-3.11.9-linux-x86_64-gnu/lib/python3.11/lib-dynload',
 '',
 '/opt/app/notebooks/abhishek/active_gliner/.venv/lib/python3.11/site-packages',
 '/tmp/tmpxous61wc',
 '/opt/app/notebooks/abhishek/active_gliner/src']

In [7]:
from config.settings import Settings
settings = Settings()

print(f"Settings cache_dir: {settings.cache_dir}")
print(f"Cache absolute path: {settings.cache_dir.resolve()}")
print(f"Does cache dir contain 'notebooks': {'notebooks' in str(settings.cache_dir)}")
settings

Settings cache_dir: /opt/app/notebooks/abhishek/active_gliner/cache
Cache absolute path: /opt/app/notebooks/abhishek/active_gliner/cache
Does cache dir contain 'notebooks': True


Settings(seed=42, batch_size=8, model=knowledgator/modern-gliner-bi-large-v1.0)

In [8]:
print("=== Integration Test ===")
from utils.logging import setup_logging
from utils.reproducibility import set_all_seeds
from utils.device import setup_device

# Complete setup like your original code
settings = Settings()
settings.setup()  # Apply environment and create directories

logger = setup_logging(log_dir=str(settings.logs_dir))
set_all_seeds(seed=settings.global_seed, logger=logger)
device = setup_device(logger=logger)

logger.info("All modules integrated successfully!")
print(f"Final setup: seed={settings.global_seed}, device={device}, batch_size={settings.batch_size}")



INFO:ActiveLearning:================================================================================
INFO:ActiveLearning:ACTIVE LEARNING PIPELINE WITH PROPER TRAIN/TEST SEPARATION
INFO:ActiveLearning:================================================================================
INFO:ActiveLearning:Log file: /opt/app/notebooks/abhishek/active_gliner/logs/ActiveLearning_20250929_123231.log
INFO:ActiveLearning:Setting all seeds to 42 for reproducibility...
INFO:ActiveLearning:Using device: cuda
INFO:ActiveLearning:CUDA version: 12.8
INFO:ActiveLearning:Number of GPUs visible: 1
INFO:ActiveLearning:Current GPU: 0
INFO:ActiveLearning:GPU Name: NVIDIA GeForce RTX 3090
INFO:ActiveLearning:GPU Memory: 23.6 GB
INFO:ActiveLearning:All modules integrated successfully!


=== Integration Test ===
Final setup: seed=42, device=cuda, batch_size=8


In [9]:
import json
from data.loader import load_mit_dataset


with open(r"../results/high_mse_2500_examples.json",mode="r") as file:
    low_n=json.load(file)


print(low_n[0])


# Load FULL test data for evaluation
test_data_path = settings.data_path / settings.test_file
labels_path = settings.data_path / settings.labels_file

if not (test_data_path.exists() and labels_path.exists()):
    raise FileNotFoundError("Test data or labels file not found!")

test_data, entity_types = load_mit_dataset(str(test_data_path), str(labels_path), "test")
logger.info(f"📊 Loaded FULL test data: {len(test_data)} examples, {len(entity_types)} entity types")


INFO:ActiveLearning:📊 Loaded FULL test data: 2442 examples, 12 entity types


{'tokenized_text': ['the', 'african', 'queen'], 'ner': [[0, 2, 'title']], 'predictions': [[2, 2, 'title']], 'scores': [0.500488817691803]}
Loading test data from: /opt/app/notebooks/abhishek/active_gliner/data/mit-movie/test.json
Processed 2442 examples
Entity types: ['genre', 'year', 'plot', 'average ratings', 'actor', 'title', 'song', 'character', 'rating', 'review', 'director', 'trailer']


In [13]:
from generation.enc_api_label import LabelGenerator, QuotaExceededException
from dotenv import load_dotenv
load_dotenv() 


n_examples=5
label_cache = []
label_generator = LabelGenerator(model_name="qwen-3-235b-a22b-thinking-2507")
train_subset = low_n[:n_examples]

llm_labeled_data = label_generator.generate(
    low_n_examples=train_subset,
    num_samples=n_examples,
    entity_types=entity_types,
    label_cache=label_cache,
    verbose=True
)






INFO:httpx:HTTP Request: GET https://api.cerebras.ai/v1/tcp_warming "HTTP/1.1 200 OK"
INFO:ActiveLearning:Enhanced Label Generator initialized: qwen-3-235b-a22b-thinking-2507
INFO:ActiveLearning:Cache directory: ../results/data
INFO:ActiveLearning:============================================================
INFO:ActiveLearning:ENHANCED CEREBRAS API LABEL GENERATION
INFO:ActiveLearning:============================================================
INFO:ActiveLearning:Model: qwen-3-235b-a22b-thinking-2507
INFO:ActiveLearning:Context limit: 65,536 tokens
INFO:ActiveLearning:Entity types: ['genre', 'year', 'plot', 'average ratings', 'actor', 'title', 'song', 'character', 'rating', 'review', 'director', 'trailer']
INFO:ActiveLearning:Low confidence examples available: 5
INFO:ActiveLearning:Target labels: 5
INFO:ActiveLearning:Current cache size: 0
INFO:ActiveLearning:Rate limits: 30 req/min, 60,000 tokens/min
INFO:ActiveLearning:No existing cache found, starting fresh
INFO:ActiveLearning:Need

Converting 4 synthetic examples to NER format...
Conversion completed: 4 examples, 0 errors
